# 🔥 DDPM — Denoising Diffusion Probabilistic Model
### Assignment No. 4 | Generative AI (AI4009) | Spring 2026
#### National University of Computer and Emerging Sciences
---
**Architecture:** U-Net with Residual Blocks + Self-Attention  
**Dataset:** CelebA-HQ-256  
**Training:** Mixed Precision · DataParallel (Dual T4)  
**Deliverables:** Forward/Reverse Diffusion · Generation · Reconstruction · PSNR/SSIM · Gradio App

In [ ]:
!pip install -q gradio==4.44.0 scikit-image
print('Dependencies installed')

In [ ]:
import os, json
KAGGLE_CREDS = {'username': 'nooreadanadan', 'key': 'ba7c9108013b2b7a6bff0bce0bc9f13a'}
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
with open(f'{kaggle_dir}/kaggle.json', 'w') as _f:
    json.dump(KAGGLE_CREDS, _f)
os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)
print('Downloading CelebA-HQ dataset...')
os.system('kaggle datasets download -d denislukovnikov/celebahq256-images-only --quiet')
os.system('unzip -q celebahq256-images-only.zip -d ./celeba_data')
import glob
n = len(glob.glob('./celeba_data/**/*.jpg', recursive=True))
print(f'Found {n} images. Dataset ready!')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import make_grid, save_image
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import glob, math, time, os
from tqdm.auto import tqdm
from skimage.metrics import peak_signal_noise_ratio as psnr_metric
from skimage.metrics import structural_similarity as ssim_metric

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
for _i in range(torch.cuda.device_count()):
    print(f'  GPU {_i} : {torch.cuda.get_device_name(_i)}')

In [ ]:
class Config:
    # ── Image / Data ─────────────────────────────────────────────
    image_size   = 64          # 64x64 → ~4x fewer pixels than 128; ~5–10 min training
    channels     = 3
    max_samples  = 10000       # more data → better quality
    # ── Training ─────────────────────────────────────────────────
    batch_size   = 32
    num_epochs   = 10          # more passes on lightweight images
    lr           = 2e-4
    weight_decay = 1e-4
    # ── Diffusion ─────────────────────────────────────────────────
    T            = 300
    beta_start   = 1e-4
    beta_end     = 0.02
    # ── Misc ──────────────────────────────────────────────────────
    num_workers  = 4
    device       = 'cuda' if torch.cuda.is_available() else 'cpu'
    save_dir     = './outputs'
    seed         = 42

cfg = Config()
os.makedirs(cfg.save_dir, exist_ok=True)
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
print(f'Device  : {cfg.device}  |  GPUs: {torch.cuda.device_count()}')
print(f'Image   : {cfg.image_size}x{cfg.image_size}  |  T={cfg.T}  |  Epochs={cfg.num_epochs}')
print(f'Samples : {cfg.max_samples}  |  Batch={cfg.batch_size}  |  Steps/epoch≈{cfg.max_samples//cfg.batch_size}')
print(f'Est. training time on T4: ~{cfg.num_epochs*(cfg.max_samples//cfg.batch_size)*0.10/60:.1f}–{cfg.num_epochs*(cfg.max_samples//cfg.batch_size)*0.15/60:.1f} min')


In [ ]:
class CelebADataset(Dataset):
    def __init__(self, root, transform=None, max_samples=5000):
        files = []
        for ext in ['*.jpg', '*.jpeg', '*.png']:
            files += glob.glob(os.path.join(root, '**', ext), recursive=True)
        self.files = sorted(files)[:max_samples]
        self.transform = transform
        print(f'  Found {len(self.files)} images')

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img

transform = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size),
                      interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5])
])

print('Loading dataset...')
dataset    = CelebADataset('./celeba_data', transform=transform,
                            max_samples=cfg.max_samples)
dataloader = DataLoader(dataset, batch_size=cfg.batch_size, shuffle=True,
                        num_workers=cfg.num_workers, pin_memory=True, drop_last=True)
print(f'  Batches/epoch: {len(dataloader)}')
sample_batch = next(iter(dataloader))
print(f'  Batch shape : {sample_batch.shape}')

In [ ]:
class DiffusionSchedule:
    def __init__(self, T, beta_start, beta_end, device):
        self.T = T; self.device = device
        # beta schedule
        self.betas           = torch.linspace(beta_start, beta_end, T, device=device)
        self.alphas          = 1.0 - self.betas
        self.alpha_bar       = torch.cumprod(self.alphas, dim=0)
        self.alpha_bar_prev  = F.pad(self.alpha_bar[:-1], (1, 0), value=1.0)
        # q_sample precomputes
        self.sqrt_ab         = torch.sqrt(self.alpha_bar)
        self.sqrt_one_m_ab   = torch.sqrt(1.0 - self.alpha_bar)
        # reverse precomputes
        self.sqrt_recip_a    = torch.sqrt(1.0 / self.alphas)
        self.post_var        = (self.betas *
                                (1.0 - self.alpha_bar_prev) /
                                (1.0 - self.alpha_bar))

    def q_sample(self, x0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x0)
        s  = self.sqrt_ab[t][:, None, None, None]
        sm = self.sqrt_one_m_ab[t][:, None, None, None]
        return s * x0 + sm * noise, noise

    def p_sample(self, model, xt, t_i):
        t_b = torch.full((xt.shape[0],), t_i, device=self.device, dtype=torch.long)
        with torch.no_grad():
            with torch.cuda.amp.autocast():
                eps = model(xt, t_b)
        coef  = self.betas[t_i] / self.sqrt_one_m_ab[t_i]
        mean  = self.sqrt_recip_a[t_i] * (xt - coef * eps)
        if t_i > 0:
            noise = torch.randn_like(xt)
            var   = torch.sqrt(self.post_var[t_i])
        else:
            noise, var = 0.0, 0.0
        return mean + var * noise

schedule = DiffusionSchedule(cfg.T, cfg.beta_start, cfg.beta_end, cfg.device)
print(f'Schedule ready | beta: {cfg.beta_start} -> {cfg.beta_end} | T={cfg.T}')
print(f'  ab_0={schedule.alpha_bar[0]:.4f}  ab_mid={schedule.alpha_bar[cfg.T//2]:.4f}  ab_T={schedule.alpha_bar[-1]:.4f}')

In [ ]:
# ── Sinusoidal Time Embedding ─────────────────────────────────────
class SinEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        half   = self.dim // 2
        freqs  = torch.exp(-math.log(10000) *
                           torch.arange(half, device=t.device) / (half - 1))
        args   = t[:, None].float() * freqs[None, :]
        return torch.cat([args.sin(), args.cos()], dim=-1)

# ── Residual Block with AdaGN time conditioning ───────────────────
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, t_dim, dropout=0.1):
        super().__init__()
        self.n1     = nn.GroupNorm(8, in_ch)
        self.c1     = nn.Conv2d(in_ch,  out_ch, 3, padding=1)
        self.n2     = nn.GroupNorm(8, out_ch)
        self.c2     = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.t_proj = nn.Linear(t_dim, out_ch * 2)
        self.drop   = nn.Dropout(dropout)
        self.skip   = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, t):
        h       = self.c1(F.silu(self.n1(x)))
        sc, sh  = self.t_proj(F.silu(t)).chunk(2, dim=-1)
        h       = h * (1 + sc[:,:,None,None]) + sh[:,:,None,None]
        h       = self.c2(self.drop(F.silu(self.n2(h))))
        return h + self.skip(x)

# ── Self-Attention Block ──────────────────────────────────────────
class AttnBlock(nn.Module):
    def __init__(self, ch, heads=4):
        super().__init__()
        self.norm = nn.GroupNorm(8, ch)
        self.attn = nn.MultiheadAttention(ch, heads, batch_first=True)
    def forward(self, x):
        b, c, h, w = x.shape
        res = x
        x   = self.norm(x).view(b, c, h*w).transpose(1, 2)
        x,_ = self.attn(x, x, x)
        return x.transpose(1,2).view(b, c, h, w) + res

print('Blocks defined: SinEmb, ResBlock, AttnBlock')

In [ ]:
class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, t_dim, attn=False):
        super().__init__()
        self.r1   = ResBlock(in_ch,  out_ch, t_dim)
        self.r2   = ResBlock(out_ch, out_ch, t_dim)
        self.attn = AttnBlock(out_ch) if attn else nn.Identity()
        self.down = nn.Conv2d(out_ch, out_ch, 4, 2, 1)
    def forward(self, x, t):
        x    = self.r2(self.r1(x, t), t)
        x    = self.attn(x)
        return self.down(x), x          # (downsampled, skip)

class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, t_dim, attn=False):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, in_ch, 4, 2, 1)
        self.r1   = ResBlock(in_ch + skip_ch, out_ch, t_dim)
        self.r2   = ResBlock(out_ch, out_ch, t_dim)
        self.attn = AttnBlock(out_ch) if attn else nn.Identity()
    def forward(self, x, skip, t):
        x = torch.cat([self.up(x), skip], dim=1)
        return self.attn(self.r2(self.r1(x, t), t))

print('DownBlock & UpBlock defined')

In [ ]:
class UNet(nn.Module):
    def __init__(self, in_ch=3, base=64, t_dim=256):
        super().__init__()
        # Time MLP
        self.t_mlp = nn.Sequential(
            SinEmb(base), nn.Linear(base, t_dim), nn.SiLU(), nn.Linear(t_dim, t_dim))
        # Stem
        self.stem = nn.Conv2d(in_ch, base, 3, padding=1)
        # Encoder: 64->128->256->256
        self.d1 = DownBlock(base,    base*2, t_dim)            # 128->64
        self.d2 = DownBlock(base*2,  base*4, t_dim)            # 64->32
        self.d3 = DownBlock(base*4,  base*4, t_dim, attn=True) # 32->16
        # Bottleneck
        self.m1 = ResBlock(base*4, base*4, t_dim)
        self.ma = AttnBlock(base*4)
        self.m2 = ResBlock(base*4, base*4, t_dim)
        # Decoder: 256->256->128->64
        self.u1 = UpBlock(base*4, base*4, base*4, t_dim, attn=True)
        self.u2 = UpBlock(base*4, base*4, base*2, t_dim)
        self.u3 = UpBlock(base*2, base*2, base,   t_dim)
        # Head
        self.head = nn.Sequential(
            nn.GroupNorm(8, base), nn.SiLU(), nn.Conv2d(base, in_ch, 1))

    def forward(self, x, t):
        te     = self.t_mlp(t)              # (B, t_dim)
        x      = self.stem(x)              # B, 64, 128,128
        x, s1  = self.d1(x, te)           # B,128,  64, 64 | skip B,128
        x, s2  = self.d2(x, te)           # B,256,  32, 32 | skip B,256
        x, s3  = self.d3(x, te)           # B,256,  16, 16 | skip B,256
        x      = self.m2(self.ma(self.m1(x, te)), te)
        x      = self.u1(x, s3, te)       # B,256,  32, 32
        x      = self.u2(x, s2, te)       # B,128,  64, 64
        x      = self.u3(x, s1, te)       # B, 64, 128,128
        return self.head(x)               # B,  3, 128,128

model  = UNet(in_ch=cfg.channels, base=64, t_dim=256)
n_par  = sum(p.numel() for p in model.parameters())
print(f'U-Net | Parameters: {n_par/1e6:.2f}M')
if torch.cuda.device_count() > 1:
    print(f'DataParallel across {torch.cuda.device_count()} GPUs')
    model = nn.DataParallel(model)
model = model.to(cfg.device)
# Sanity check
with torch.no_grad():
    _o = model(torch.randn(2,3,cfg.image_size,cfg.image_size,device=cfg.device),
               torch.randint(0,cfg.T,(2,),device=cfg.device))
print(f'Forward pass OK -> output shape: {_o.shape}')

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(),
                               lr=cfg.lr, weight_decay=cfg.weight_decay)
total_steps = cfg.num_epochs * len(dataloader)
lr_sched    = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, total_steps)
scaler      = torch.cuda.amp.GradScaler()
print(f'Optimizer : AdamW lr={cfg.lr}')
print(f'Scheduler : CosineAnnealing over {total_steps} steps')

In [ ]:
train_losses = []
start_time   = time.time()
best_loss    = float('inf')

print('=' * 60)
print(f' Training: {cfg.num_epochs} epochs x {len(dataloader)} steps')
print('=' * 60)

for epoch in range(1, cfg.num_epochs + 1):
    model.train()
    epoch_loss = 0.0
    pbar = tqdm(dataloader, desc=f'Epoch {epoch:02d}/{cfg.num_epochs}')

    for batch in pbar:
        batch = batch.to(cfg.device, non_blocking=True)
        t     = torch.randint(0, cfg.T, (batch.size(0),), device=cfg.device).long()
        noise = torch.randn_like(batch)
        noisy, _ = schedule.q_sample(batch, t, noise)

        with torch.cuda.amp.autocast():
            pred = model(noisy, t)
            loss = F.mse_loss(pred, noise)

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        lr_sched.step()

        epoch_loss += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.4f}',
                         lr=f'{lr_sched.get_last_lr()[0]:.1e}')

    avg = epoch_loss / len(dataloader)
    train_losses.append(avg)
    elapsed = (time.time() - start_time) / 60
    print(f'  Epoch {epoch:02d} | Loss={avg:.5f} | {elapsed:.1f} min')

    if avg < best_loss:
        best_loss = avg
        mdl = model.module if isinstance(model, nn.DataParallel) else model
        torch.save(mdl.state_dict(), f'{cfg.save_dir}/ddpm_best.pth')

total_min = (time.time() - start_time) / 60
print(f'Done in {total_min:.1f} min  |  Best loss: {best_loss:.5f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ep = range(1, len(train_losses)+1)
ax.plot(ep, train_losses, 'o-', color='#7C3AED', lw=2.5, ms=8,
        markerfacecolor='white', markeredgewidth=2, markeredgecolor='#7C3AED')
ax.fill_between(ep, train_losses, alpha=0.15, color='#7C3AED')
ax.set_xlabel('Epoch', fontsize=13)
ax.set_ylabel('MSE Loss', fontsize=13)
ax.set_title('DDPM Training Loss', fontsize=16, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{cfg.save_dir}/training_loss.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Final loss: {train_losses[-1]:.5f}')

In [ ]:
def to_np(t):
    return ((t.clamp(-1,1)+1)/2*255).permute(1,2,0).cpu().numpy().astype(np.uint8)

def to_pil(t):
    return Image.fromarray(to_np(t))

def show_grid(images, title, cols=5, save=True):
    n    = min(len(images), cols)
    grid = make_grid([img.clamp(-1,1).cpu() for img in images[:n]],
                     nrow=n, normalize=True, value_range=(-1,1))
    fig, ax = plt.subplots(figsize=(15, 3.5))
    ax.imshow(grid.permute(1,2,0).numpy())
    ax.set_title(title, fontsize=15, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    if save:
        plt.savefig(f'{cfg.save_dir}/{title[:40].replace(" ","_")}.png',
                    dpi=150, bbox_inches='tight')
    plt.show()

print('Helpers ready: to_np, to_pil, show_grid')

In [ ]:
# Part 2 — Forward Process Visualization
model.eval()
orig = next(iter(dataloader))[:1].to(cfg.device)

# Dynamic timesteps – works for any cfg.T value
n_vis    = 6
step_ids = [int(x) for x in np.linspace(0, cfg.T - 1, n_vis)]
lbls     = [f't={s}' for s in step_ids]
lbls[0]  = 'Original (t=0)'
lbls[-1] = f'Pure Noise (t={cfg.T-1})'

fwd_imgs = []
for s in step_ids:
    noisy, _ = schedule.q_sample(orig, torch.tensor([s], device=cfg.device))
    fwd_imgs.append(noisy[0])

fig, axes = plt.subplots(1, n_vis, figsize=(18, 3.5))
for ax, img, lbl in zip(axes, fwd_imgs, lbls):
    ax.imshow(to_np(img))
    ax.set_title(lbl, fontsize=10)
    ax.axis('off')
plt.suptitle('Forward Diffusion Process — Progressive Noising',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{cfg.save_dir}/forward_diffusion.png', dpi=150, bbox_inches='tight')
plt.show()
print('Forward diffusion steps saved.')


In [ ]:
@torch.no_grad()
def ddpm_sample(n=1, n_inter=10, start_x=None, start_t=None):
    """
    Full generation : start_x=None, start_t=None  → sample from pure Gaussian noise
    Reconstruction  : start_x=<noisy tensor>, start_t=<int>  → denoise from given state
    """
    model.eval()
    if start_x is not None and start_t is not None:
        x       = start_x.clone()
        T_start = int(start_t)
    else:
        x       = torch.randn(n, cfg.channels, cfg.image_size, cfg.image_size,
                              device=cfg.device)
        T_start = cfg.T - 1

    steps  = [x.clone().cpu()]
    save_t = set(np.linspace(T_start, 0, min(n_inter, T_start + 1), dtype=int))
    for t_i in tqdm(reversed(range(T_start + 1)), total=T_start + 1,
                    desc='Reverse diffusion', leave=False):
        x = schedule.p_sample(model, x, t_i)
        if t_i in save_t:
            steps.append(x.clone().cpu())
    steps.append(x.clone().cpu())
    return x, steps

print('Sampler ready  (supports full generation + partial reconstruction)')


In [ ]:
print('Generating 5 images from pure Gaussian noise...')
t0 = time.time()
gen_imgs, gen_steps = ddpm_sample(n=5, n_inter=10)
print(f'Done in {time.time()-t0:.1f}s')

show_grid(list(gen_imgs), 'Generated Images from Pure Noise', cols=5)
for i, img in enumerate(gen_imgs):
    to_pil(img).save(f'{cfg.save_dir}/generated_{i+1}.png')
print('5 generated images saved.')

In [ ]:
picks   = np.linspace(0, len(gen_steps)-1, 5, dtype=int)
rev_vis = [gen_steps[i][0] for i in picks]

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
step_lbls = ['Pure Noise', 'Step 1/4', 'Step 2/4', 'Step 3/4', 'Final Image']
for ax, img, lbl in zip(axes, rev_vis, step_lbls):
    ax.imshow(to_np(img))
    ax.set_title(lbl, fontsize=12)
    ax.axis('off')
plt.suptitle('Reverse Diffusion — Progressive Denoising',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{cfg.save_dir}/reverse_diffusion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Part 5 — Image Reconstruction
# Strategy: forward-diffuse target to T//2, then denoise back to 0.
# This gives meaningful PSNR/SSIM – the model is genuinely recovering signal.
target_batch = next(iter(dataloader))[:1].to(cfg.device)
target_t     = target_batch[0]

# Step 1 – partially corrupt the target
recon_t_level = cfg.T // 2
t_tensor      = torch.full((1,), recon_t_level, device=cfg.device, dtype=torch.long)
noisy_target, _ = schedule.q_sample(target_batch, t_tensor)

# Step 2 – denoise from that noisy state back to t=0
print(f'Reconstructing: t={recon_t_level} → t=0 ...')
recon_batch, recon_steps = ddpm_sample(
    n=1, n_inter=6,
    start_x=noisy_target,
    start_t=recon_t_level
)
recon_t = recon_batch[0]

# Step 3 – compute metrics
orig_np  = to_np(target_t)
recon_np = to_np(recon_t)
psnr_v   = psnr_metric(orig_np, recon_np, data_range=255)
ssim_v   = ssim_metric(orig_np, recon_np, data_range=255,
                        channel_axis=2, win_size=7)

print(f'PSNR : {psnr_v:.2f} dB')
print(f'SSIM : {ssim_v:.4f}')

# Step 4 – visualise: original | noisy halfway | reconstructed
noisy_np = to_np(noisy_target[0])
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(orig_np)
axes[0].set_title('Original Image', fontsize=13)
axes[0].axis('off')
axes[1].imshow(noisy_np)
axes[1].set_title(f'Noisy Input (t={recon_t_level})', fontsize=13)
axes[1].axis('off')
axes[2].imshow(recon_np)
axes[2].set_title(
    f'Reconstructed\nPSNR={psnr_v:.2f} dB  SSIM={ssim_v:.4f}', fontsize=12)
axes[2].axis('off')
plt.suptitle('Image Reconstruction: Partial Noising → Denoising',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{cfg.save_dir}/reconstruction.png', dpi=150, bbox_inches='tight')
plt.show()
to_pil(target_t).save(f'{cfg.save_dir}/target.png')
to_pil(recon_t).save(f'{cfg.save_dir}/reconstructed.png')
print('Reconstruction saved.')


In [ ]:
import gradio as gr
import numpy as np
from PIL import Image as PILImage

def tensor_to_pil(t):
    arr = ((t.clamp(-1,1)+1)/2*255).permute(1,2,0).cpu().numpy().astype(np.uint8)
    return PILImage.fromarray(arr)

def run_generate(num_imgs, progress=gr.Progress(track_tqdm=True)):
    gen, steps = ddpm_sample(n=int(num_imgs), n_inter=10)
    out   = [tensor_to_pil(gen[i]) for i in range(int(num_imgs))]
    inter = [tensor_to_pil(steps[j][0]) for j in range(len(steps))]
    return out, inter

def run_recon(progress=gr.Progress(track_tqdm=True)):
    tgt_b    = next(iter(dataloader))[:1].to(cfg.device)
    tgt      = tgt_b[0]
    # Partial-noise reconstruction: noise to T//2, then denoise
    t_level  = cfg.T // 2
    t_tensor = torch.full((1,), t_level, device=cfg.device, dtype=torch.long)
    noisy_tgt, _ = schedule.q_sample(tgt_b, t_tensor)
    rec_b, rsteps = ddpm_sample(n=1, n_inter=6, start_x=noisy_tgt, start_t=t_level)
    rec      = rec_b[0]
    pv = psnr_metric(to_np(tgt), to_np(rec), data_range=255)
    sv = ssim_metric(to_np(tgt), to_np(rec), data_range=255, channel_axis=2, win_size=7)
    mtxt = (f'### Evaluation Metrics (noise level t={t_level}/{cfg.T})\n'
            f'| Metric | Value |\n|--------|-------|\n'
            f'| **PSNR** | `{pv:.2f} dB` |\n| **SSIM** | `{sv:.4f}` |')
    rvis = [tensor_to_pil(rsteps[j][0]) for j in range(len(rsteps))]
    return tensor_to_pil(tgt), tensor_to_pil(rec), mtxt, rvis

CSS = '''
.gradio-container {
    background: linear-gradient(135deg, #0d0221 0%, #1a0533 40%, #0d1b4b 100%) !important;
    font-family: Inter, Segoe UI, sans-serif;
}
.main-hdr {
    text-align: center; padding: 28px 20px;
    background: linear-gradient(135deg, #7928CA 0%, #FF0080 50%, #FF6B6B 100%);
    border-radius: 20px; margin-bottom: 22px;
    box-shadow: 0 8px 40px rgba(121,40,202,0.55); color: white;
}
.main-hdr h1 { font-size: 2.3em; font-weight: 800; margin: 0; letter-spacing: -1px; }
.main-hdr p  { font-size: 1em; opacity: 0.88; margin: 8px 0 0; }
.card {
    background: rgba(255,255,255,0.04) !important;
    border: 1px solid rgba(255,255,255,0.12) !important;
    border-radius: 16px !important; backdrop-filter: blur(12px);
}
.gr-button-primary {
    background: linear-gradient(90deg, #7928CA, #FF0080) !important;
    border: none !important; border-radius: 12px !important;
    font-size: 1.05em !important; font-weight: 700 !important;
    padding: 12px 26px !important;
    box-shadow: 0 4px 20px rgba(121,40,202,0.45) !important;
    transition: all 0.2s !important;
}
.gr-button-primary:hover { transform: translateY(-2px) !important;
    box-shadow: 0 8px 32px rgba(121,40,202,0.65) !important; }
.tab-nav button.selected {
    background: linear-gradient(90deg,#7928CA,#FF0080) !important;
    color: white !important; border-radius: 10px 10px 0 0 !important;
}
'''

with gr.Blocks(css=CSS, title='DDPM Image Generator') as demo:
    gr.HTML("""
    <div class='main-hdr'>
        <h1>&#x1F52E; DDPM &mdash; Diffusion Image Generator</h1>
        <p>Denoising Diffusion Probabilistic Model &nbsp;&bull;&nbsp;
           U-Net Architecture &nbsp;&bull;&nbsp; CelebA-HQ &nbsp;&bull;&nbsp;
           Assignment No. 4 &middot; AI4009 &middot; Spring 2026</p>
    </div>
    """)

    with gr.Tabs():

        with gr.TabItem('&#x1F3A8;  Generate Images'):
            gr.Markdown('### Generate brand-new faces from pure Gaussian noise')
            with gr.Row():
                with gr.Column(scale=1, elem_classes='card'):
                    gr.Markdown('#### Settings')
                    n_sl   = gr.Slider(1, 8, value=5, step=1, label='Number of images')
                    g_btn  = gr.Button('Lets Go! Generate', variant='primary')
                with gr.Column(scale=3):
                    gr.Markdown('#### Generated Images')
                    g_gal  = gr.Gallery(label='Final Outputs', columns=5,
                                        height=300, object_fit='cover')
                    gr.Markdown('#### Denoising Steps  (Noise → Image)')
                    s_gal  = gr.Gallery(label='Intermediate Steps', columns=8,
                                        height=180, object_fit='cover')
            g_btn.click(run_generate, inputs=[n_sl], outputs=[g_gal, s_gal])

        with gr.TabItem('&#x1F52C;  Reconstruct'):
            gr.Markdown('### Reconstruct a target image &mdash; with PSNR & SSIM metrics')
            with gr.Row():
                with gr.Column(scale=1, elem_classes='card'):
                    gr.Markdown('#### Controls')
                    r_btn  = gr.Button('Run Reconstruction', variant='primary')
                    m_md   = gr.Markdown('_Metrics appear here after running_')
                with gr.Column(scale=3):
                    with gr.Row():
                        t_img = gr.Image(label='Target Image',    height=280)
                        r_img = gr.Image(label='Generated Image',  height=280)
                    gr.Markdown('#### Denoising Steps')
                    rs_gal = gr.Gallery(label='Reverse Steps', columns=7,
                                        height=180, object_fit='cover')
            r_btn.click(run_recon, inputs=[], outputs=[t_img, r_img, m_md, rs_gal])

        with gr.TabItem('&#x1F4CA;  Info'):
            gr.Markdown(f'''
## Model Architecture
| Component | Details |
|-----------|----------|
| **Model** | DDPM |
| **Backbone** | U-Net (Residual + Attention) |
| **Channels** | 64 -> 128 -> 256 |
| **Time Embedding** | Sinusoidal + 2-layer MLP |
| **Image Size** | {cfg.image_size}x{cfg.image_size} |
| **Timesteps T** | {cfg.T} |
| **Beta schedule** | Linear {cfg.beta_start} to {cfg.beta_end} |
| **Training** | {cfg.num_epochs} epochs, AdamW, Mixed Precision, DataParallel |
| **Loss** | MSE (pred noise vs real noise) |
''')

demo.launch(share=True, quiet=False)
